# 01 — Data quality report: Berkeley Earth city temperatures

Phase 1 QA for `GlobalLandTemperaturesByCity.csv` after ingestion into
DuckDB (`data/processed/climate.duckdb`, table `city_temps`).

Covered:

1. Row counts and basic integrity
2. Null rate of `AverageTemperature` by decade
3. Duplicate `(City, dt)` check
4. Per-city coverage in the default analysis window
5. Station count over time
6. Analysis-window decision

All analysis logic is imported from `src/` — this notebook only queries,
summarizes, and plots. Regenerate with:

```
uv run jupyter nbconvert --to notebook --execute --inplace notebooks/01_data_quality.ipynb
```

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():  # kernel started inside notebooks/
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
import plotly.express as px
import plotly.io as pio

from src.cleaning import DEFAULT_END, DEFAULT_START, coverage_by_city
from src.data_io import (
    CITY_TABLE,
    DEFAULT_DB_PATH,
    city_csv_path,
    download_raw_data,
    load_city_temperatures,
)

# JSON-only plotly output: renders in JupyterLab/VS Code, keeps the file small.
pio.renderers.default = "plotly_mimetype"

print(f"default analysis window: {DEFAULT_START} .. {DEFAULT_END}")

default analysis window: 1950-01-01 .. 2013-09-01


## 0. Ensure data is downloaded and ingested

Idempotent: the download skips files already in `data/raw/`, and ingestion
runs only if the `city_temps` table is missing.

In [2]:
try:
    csv_path = city_csv_path()
except FileNotFoundError:
    download_raw_data()
    csv_path = city_csv_path()
print("city CSV:", csv_path)


def _city_table_ready() -> bool:
    if not DEFAULT_DB_PATH.exists():
        return False
    c = duckdb.connect(str(DEFAULT_DB_PATH), read_only=True)
    try:
        n = c.execute(
            "SELECT count(*) FROM duckdb_tables() WHERE table_name = ?",
            [CITY_TABLE],
        ).fetchone()[0]
        return n > 0
    finally:
        c.close()


if not _city_table_ready():
    print(load_city_temperatures(csv_path, DEFAULT_DB_PATH))

con = duckdb.connect(str(DEFAULT_DB_PATH), read_only=True)

city CSV: /Users/malek/Library/Mobile Documents/com~apple~CloudDocs/personal docs/climate-inequality/data/raw/climate-change-earth-surface-temperature-data/GlobalLandTemperaturesByCity.csv


## 1. Row counts and basics

In [3]:
overview = con.execute(f"""
    SELECT
        count(*)                                   AS n_rows,
        count(DISTINCT (City, Country))            AS n_city_country,
        count(DISTINCT Country)                    AS n_countries,
        min(dt)                                    AS dt_min,
        max(dt)                                    AS dt_max,
        count(*) - count(AverageTemperature)       AS n_null_temp,
        round(100.0 * (count(*) - count(AverageTemperature)) / count(*), 2)
                                                   AS pct_null_temp
    FROM {CITY_TABLE}
""").df()
overview

,n_rows,n_city_country,n_countries,dt_min,dt_max,n_null_temp,pct_null_temp
0,8599212,3490,159,1743-11-01,2013-09-01,364130,4.23


## 2. Null rate of `AverageTemperature` by decade

In [4]:
nulls_by_decade = con.execute(f"""
    SELECT
        (year(dt) // 10) * 10                      AS decade,
        count(*)                                   AS n_rows,
        count(*) - count(AverageTemperature)       AS n_null,
        round(100.0 * (count(*) - count(AverageTemperature)) / count(*), 2)
                                                   AS pct_null
    FROM {CITY_TABLE}
    GROUP BY decade
    ORDER BY decade
""").df()
nulls_by_decade

,decade,n_rows,n_null,pct_null
0,1740,52244,43066,82.43
1,1750,89274,9578,10.73
2,1760,93896,4461,4.75
3,1770,96062,4077,4.24
4,1780,103095,5659,5.49
5,1790,126334,8763,6.94
6,1800,157095,19305,12.29
7,1810,168727,20196,11.97
8,1820,227163,26939,11.86
9,1830,280146,49716,17.75


In [5]:
fig = px.bar(
    nulls_by_decade,
    x="decade",
    y="pct_null",
    title="Null rate of AverageTemperature by decade",
    labels={"decade": "decade", "pct_null": "% null"},
)
fig.add_vline(x=1950, line_dash="dash", line_color="gray")
fig.show()

## 3. Duplicate `(City, dt)` check

In [6]:
key_check = con.execute(f"""
    SELECT
        count(*)                                                 AS n_rows,
        count(DISTINCT (City, dt))                               AS distinct_city_dt,
        count(DISTINCT (City, Country, dt))                      AS distinct_city_country_dt,
        count(DISTINCT (City, Country, Latitude, Longitude, dt)) AS distinct_full_key,
        count(DISTINCT (City, Country))                          AS distinct_city_country,
        count(DISTINCT (City, Country, Latitude, Longitude))     AS distinct_locations
    FROM {CITY_TABLE}
""").df()
key_check.T.rename(columns={0: "value"})

,value
n_rows,8599212
distinct_city_dt,8455054
distinct_city_country_dt,8553178
distinct_full_key,8599212
distinct_city_country,3490
distinct_locations,3510


In [7]:
n_rows = int(key_check.loc[0, "n_rows"])
city_dt = int(key_check.loc[0, "distinct_city_dt"])
cc_dt = int(key_check.loc[0, "distinct_city_country_dt"])
full_key = int(key_check.loc[0, "distinct_full_key"])
n_pairs = int(key_check.loc[0, "distinct_city_country"])
n_locations = int(key_check.loc[0, "distinct_locations"])

print(
    f"(City, dt) collides for {n_rows - city_dt:,} rows "
    "-> same-named cities in different countries (expected, not an error)."
)
print(
    f"(City, Country, dt) collides for {n_rows - cc_dt:,} rows "
    f"-> {n_locations - n_pairs} extra coordinate locations hide behind "
    "same-named (City, Country) pairs (listed below)."
)
print(
    "(City, Country, Latitude, Longitude, dt) unique:",
    "PASS" if full_key == n_rows else f"FAIL - {n_rows - full_key:,} duplicate rows",
)

dual_located = con.execute(f"""
    SELECT City, Country,
           count(DISTINCT (Latitude, Longitude)) AS n_locations,
           count(*)                              AS n_rows
    FROM {CITY_TABLE}
    GROUP BY City, Country
    HAVING count(DISTINCT (Latitude, Longitude)) > 1
    ORDER BY Country, City
""").df()
dual_located

(City, dt) collides for 144,158 rows -> same-named cities in different countries (expected, not an error).
(City, Country, dt) collides for 46,034 rows -> 20 extra coordinate locations hide behind same-named (City, Country) pairs (listed below).
(City, Country, Latitude, Longitude, dt) unique: PASS


,City,Country,n_locations,n_rows
0,Haicheng,China,2,4295
1,Haikou,China,2,4456
2,Jining,China,2,4391
3,Luoyang,China,2,4167
4,Rongcheng,China,3,6526
5,Suzhou,China,2,4146
6,Yichun,China,2,4391
7,Yingcheng,China,2,4155
8,Depok,Indonesia,2,4530
9,Taman,Indonesia,2,4530


In [8]:
name_collisions = con.execute(f"""
    SELECT City, count(DISTINCT Country) AS n_countries
    FROM {CITY_TABLE}
    GROUP BY City
    HAVING count(DISTINCT Country) > 1
    ORDER BY n_countries DESC, City
    LIMIT 10
""").df()
name_collisions

,City,n_countries
0,León,3
1,Santiago,3
2,Worcester,3
3,Alexandria,2
4,Barcelona,2
5,Bharatpur,2
6,Birmingham,2
7,Brest,2
8,Cadiz,2
9,Cambridge,2


## 4. Per-city coverage in the default window

Uses `src.cleaning.coverage_by_city`. Only three columns are pulled out of
DuckDB, with null temperatures already dropped in SQL (the non-null count
per city is unchanged by this).

Note: `coverage_by_city` groups by `(City, Country)`, so the dual-located
same-named pairs found in section 3 are double-counted and show
`coverage` ~ 2.0 — a known artifact flagged below, to be resolved by the
Phase 2 dedup decision.

In [9]:
window_obs = con.execute(f"""
    SELECT City, Country, AverageTemperature
    FROM {CITY_TABLE}
    WHERE dt BETWEEN DATE '{DEFAULT_START}' AND DATE '{DEFAULT_END}'
      AND AverageTemperature IS NOT NULL
""").df()

coverage = coverage_by_city(window_obs)
n_keep = int(coverage["keep"].sum())
n_total = int(overview.loc[0, "n_city_country"])
n_inflated = int((coverage["coverage"] > 1).sum())

print(f"cities with any in-window data:  {len(coverage):,}")
print(f"cities with zero in-window data: {n_total - len(coverage):,}")
print(
    f"pass >=90% coverage:             {n_keep:,} "
    f"({100 * n_keep / len(coverage):.1f}% of cities in window)"
)
print(
    f"coverage > 1 (dual-located same-named pairs from section 3, "
    f"double-counted): {n_inflated}"
)
coverage.sort_values("coverage").head(10)

cities with any in-window data:  3,490
cities with zero in-window data: 0
pass >=90% coverage:             3,490 (100.0% of cities in window)
coverage > 1 (dual-located same-named pairs from section 3, double-counted): 18


,City,Country,n_obs,n_possible,coverage,keep
0,A Coruña,Spain,764,765,0.998693,True
2233,Ordu,Turkey,764,765,0.998693,True
2234,Orekhovo Zuevo,Russia,764,765,0.998693,True
2236,Orkney,South Africa,764,765,0.998693,True
2238,Orléans,France,764,765,0.998693,True
2239,Oron,Nigeria,764,765,0.998693,True
2240,Orsha,Belarus,764,765,0.998693,True
2241,Orsk,Russia,764,765,0.998693,True
2242,Orumiyeh,Iran,764,765,0.998693,True
2243,Oruro,Bolivia,764,765,0.998693,True


In [10]:
fig = px.histogram(
    coverage,
    x="coverage",
    nbins=50,
    title=f"Per-city observation coverage, {DEFAULT_START[:7]} .. {DEFAULT_END[:7]}",
    labels={"coverage": "fraction of window months with data"},
)
fig.add_vline(x=0.9, line_dash="dash", line_color="red")
fig.show()

## 5. Station count over time

In [11]:
stations = con.execute(f"""
    SELECT year(dt) AS year, count(DISTINCT (City, Country)) AS n_cities
    FROM {CITY_TABLE}
    WHERE AverageTemperature IS NOT NULL
    GROUP BY year
    ORDER BY year
""").df()

fig = px.line(
    stations,
    x="year",
    y="n_cities",
    title="Cities reporting at least one non-null month, by year",
    labels={"n_cities": "reporting cities"},
)
fig.add_vline(x=1950, line_dash="dash", line_color="gray")
fig.add_vline(x=2013, line_dash="dash", line_color="gray")
fig.show()

## 6. Analysis window decision

Evidence from this report:

- Nulls in `AverageTemperature` are an 18th/19th-century problem: 82% of
  rows in the 1740s, 5–18% per decade through the 1880s, **0.00% from 1900
  through 2009**, and 1.9% in the partial 2010s (the dataset's final months
  in 2013 carry some missing values).
- The reporting-station curve (section 5) climbs through the 19th century
  and is flat at the full panel from ~1900 onward.
- Within 1950-01 .. 2013-09, per-city coverage is effectively complete:
  every (City, Country) pair retains ≥ 99.8% of the 765 window months, so
  the ≥ 90% coverage filter excludes nothing.

**Decision: keep the default window 1950-01 .. 2013-09**
(`src.cleaning.DEFAULT_START` / `DEFAULT_END`) — complete city coverage,
the period of strongest global station representativity, and the dataset
ends 2013-09.

Carried forward to Phase 2:

- 20 extra coordinate locations hide behind same-named `(City, Country)`
  pairs; group by `(City, Country, Latitude, Longitude)` when fitting
  trends, or the affected series get mixed and coverage double-counts
  (their `coverage` ~ 2.0 in section 4).
- `AverageTemperatureUncertainty` is ingested and available for weighted
  fits, or at minimum for the limitations section.

In [12]:
con.close()